# Inference Run Analysis

This notebook summarizes the outputs of a single MDM inference run.

**Purpose**
- Inspect prompts used
- Verify sample counts
- Analyze intra-prompt diversity
- Inspect global distance statistics

**Notes**
- Motion data is NOT loaded
- This notebook is safe to run on CPU
- Designed to be copied per inference run


In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

RUN_DIR = Path(".")
METRICS_DIR = RUN_DIR / "metrics"

MANIFEST_PATH = RUN_DIR / "manifest.jsonl"
FILE_LIST_PATH = RUN_DIR / "file_list.js"
PAIRWISE_PATH = METRICS_DIR / "pairwise_distances.csv"
INTRA_PATH = METRICS_DIR / "intra_prompt_diversity.csv"

print("Run directory:", RUN_DIR.resolve())


In [ ]:
manifest = []

with open(MANIFEST_PATH, "r", encoding="utf-8") as f:
    for line in f:
        manifest.append(json.loads(line))

df_manifest = pd.DataFrame(manifest)
df_manifest.head()


In [ ]:
print("Total samples:", len(df_manifest))
print("Unique prompts:", df_manifest["prompt_uid"].nunique())
print("Samples per prompt:")
df_manifest.groupby("prompt_uid")["sample_idx"].count().value_counts()


In [ ]:
df_manifest[
    ["prompt_uid", "scene", "sequence", "text", "guidance"]
].drop_duplicates().reset_index(drop=True)


In [ ]:
df_intra = pd.read_csv(INTRA_PATH)
df_intra


In [ ]:
plt.figure(figsize=(10, 4))
plt.bar(df_intra["prompt_uid"], df_intra["mean_l2"])
plt.xticks(rotation=90)
plt.ylabel("Mean L2 distance")
plt.title("Intra-prompt Motion Diversity")
plt.tight_layout()
plt.show()


In [ ]:
df_pairs = pd.read_csv(PAIRWISE_PATH)
df_pairs.head()


In [ ]:
plt.figure(figsize=(6, 4))
plt.hist(df_pairs["l2_rmse"], bins=50)
plt.xlabel("L2 RMSE")
plt.ylabel("Count")
plt.title("Global Pairwise Motion Distance Distribution")
plt.tight_layout()
plt.show()


In [ ]:
assert not df_manifest.empty
assert not df_intra.empty
assert not df_pairs.empty

print("All checks passed ✔")
